# 40.01 Обработка ТТРКГ эксперимента 2

Ноутбук строит знаковые ЭКГ-синхронизированные ансамбли канала ТТРКГ
реокардиомонитора МГТУ. Каждая последовательная запись боковой сборки остаётся
отдельным наблюдением. Точный набор из 19 независимых записей принимает `10.01`:
без его QC-манифеста исключённая копия 100 мм может ошибочно вернуться в анализ.

Реальный расчёт требует принятых дыхательных и ЭКГ-sidecar, операторов амплитуды
и базового импеданса, а также измеренной групповой задержки. Временная сетка
выхода исправлена на эту задержку. Внутризаписная стандартная ошибка ансамбля
не объявляется полной неопределённостью.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import numpy as np

from ttrkg_analysis import ensemble_waveform

REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def accepted_sidecars(directory, annotation_type):
    result = {}
    for path in sorted(Path(directory).glob("*.json")):
        item = json.loads(path.read_text(encoding="utf-8"))
        if item.get("annotation_type") != annotation_type:
            continue
        if item.get("qc", {}).get("status") != "accepted":
            raise RuntimeError(f"Непринятый {annotation_type} sidecar: {item.get('record_id')}")
        record_id = item.get("record_id")
        if not record_id or record_id in result:
            raise RuntimeError("Пустой или повторный record_id в sidecar")
        result[record_id] = (path, item)
    return result


def accepted_record_manifest(config, experiment_id):
    spec = config.get("record_manifest", {})
    if spec.get("status") != "accepted" or not spec.get("path"):
        raise RuntimeError("Не задан принятый QC-манифест записей")
    path = Path(spec["path"]).expanduser().resolve()
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("status") != "accepted" or manifest.get("experiment_id") != experiment_id:
        raise RuntimeError("QC-манифест не принят или относится к другому эксперименту")
    records = [item for item in manifest.get("records", []) if item.get("include") is True]
    ids = [item.get("record_id") for item in records]
    if not records or any(not value for value in ids) or len(ids) != len(set(ids)):
        raise RuntimeError("QC-манифест должен содержать непустой уникальный набор принятых записей")
    required = {
        "record_id", "subject_id", "relative_path", "input_sha256",
        "actual_active_channels", "include", "qc_status", "configuration_id",
        "montage_id", "side_montage_id", "side_size_mm",
    }
    for item in records:
        if not required.issubset(item):
            raise RuntimeError(f"В записи {item.get('record_id')} неполный QC-контракт")
        if item["qc_status"] != "accepted":
            raise RuntimeError(f"Запись не принята QC: {item['record_id']}")
        if sorted(item["actual_active_channels"]) != item["actual_active_channels"]:
            raise RuntimeError("actual_active_channels должен быть упорядоченным списком")
    return path, manifest, {item["record_id"]: item for item in records}


def require_ttrkg_operator(config):
    operator = config.get("ttrkg_analysis", {})
    required = {
        "signal_column", "base_column", "unit_scale_to_ohm", "sign", "gain",
        "pulse_calibration_status", "pulse_operator_status", "base_sign", "base_gain",
        "base_offset_ohm", "base_calibration_status", "pre_s", "post_s", "grid_step_s",
        "baseline_s", "hold_margin_s", "timing_calibration_status", "group_delay_s",
        "montage_id",
    }
    if not required.issubset(operator):
        raise RuntimeError("В ttrkg_analysis отсутствуют обязательные поля")
    if operator["pulse_calibration_status"] != "accepted" or operator["pulse_operator_status"] != "delta_impedance_ohm_accepted":
        raise RuntimeError("Не принят оператор пульсового канала ТТРКГ")
    if operator["base_calibration_status"] != "accepted":
        raise RuntimeError("Не принята абсолютная шкала базового импеданса ТТРКГ")
    if operator["timing_calibration_status"] != "accepted" or operator["group_delay_s"] is None:
        raise RuntimeError("Не принята временная калибровка и групповая задержка ТТРКГ")
    for key in ("sign", "base_sign"):
        if operator[key] not in (-1, 1):
            raise ValueError(f"{key} должен быть -1 или 1")
    for key in ("gain", "base_gain", "unit_scale_to_ohm"):
        if operator[key] is None or float(operator[key]) <= 0:
            raise ValueError(f"{key} должен быть положительным")
    if not np.isfinite(float(operator["group_delay_s"])):
        raise ValueError("group_delay_s должен быть конечным")
    if not operator["montage_id"]:
        raise ValueError("montage_id не может быть пустым")
    return operator


def synthetic_ensemble_test(label):
    time_s = np.arange(0.0, 14.0, 0.002)
    rpeaks_s = np.arange(1.0, 13.0, 1.0)
    signal = -0.015 * np.exp(-((time_s[:, None] - (rpeaks_s[None, :] + 0.28)) / 0.06) ** 2).sum(axis=1)
    grid_s = np.arange(-0.15, 0.701, 0.005)
    result = ensemble_waveform(time_s, signal, rpeaks_s, (0.5, 13.5), grid_s, (-0.12, -0.02))
    assert result["n_beats"] == len(rpeaks_s)
    assert abs(grid_s[int(np.argmin(result["mean"]))] - 0.28) <= 0.005
    print(f"{label} synthetic_self_test: passed")

synthetic_ensemble_test("40.01")


In [ ]:
if not REAL_MODE:
    print("40.01 real_data_status: blocked_until_record_manifest_sidecars_calibration_and_timing")
else:
    import pandas as pd

    config_path = Path(os.environ["KALMYKOV_EXP02_CONFIG"]).expanduser().resolve()
    config = json.loads(config_path.read_text(encoding="utf-8"))
    operator = require_ttrkg_operator(config)
    manifest_path, record_manifest, records = accepted_record_manifest(config, "exp02")
    expected_pairs = {
        (subject["subject_id"], int(size_mm))
        for subject in config["subjects"] for size_mm in subject["sizes_mm"]
    }
    actual_pairs = {(item["subject_id"], int(item["side_size_mm"])) for item in records.values()}
    if actual_pairs != expected_pairs or len(records) != int(config["expected_independent_record_count"]):
        raise RuntimeError("QC-манифест не совпадает с точным набором 19 независимых записей эксперимента 2")
    if any(item["actual_active_channels"] != [1, 2] for item in records.values()):
        raise RuntimeError("Для записи эксперимента 2 не подтверждены фактически активные каналы 1 и 2")

    data_root = Path(config["data_root"]).expanduser().resolve()
    derived_root = Path(config["derived_root"]).expanduser().resolve()
    breathing = accepted_sidecars(derived_root / "exp02" / "annotations" / "breathing", "breathing")
    ecg = accepted_sidecars(derived_root / "exp02" / "annotations" / "ecg", "ecg")
    if set(breathing) != set(records) or set(ecg) != set(records):
        raise RuntimeError("Наборы QC-манифеста, дыхательных и ЭКГ-sidecar должны точно совпадать")

    raw_grid_s = np.arange(-float(operator["pre_s"]), float(operator["post_s"]) + 0.5 * float(operator["grid_step_s"]), float(operator["grid_step_s"]))
    corrected_grid_s = raw_grid_s - float(operator["group_delay_s"])
    outputs = []
    for record_id, record in sorted(records.items()):
        breath_path, breath = breathing[record_id]
        ecg_path, ecg_item = ecg[record_id]
        if breath.get("record_id") != record_id or ecg_item.get("record_id") != record_id:
            raise RuntimeError("record_id sidecar не совпал с QC-манифестом")
        if breath.get("subject_id") != record["subject_id"] or int(breath.get("size_mm")) != int(record["side_size_mm"]):
            raise RuntimeError("Доброволец или размер sidecar не совпал с QC-манифестом")
        if breath["input"]["sha256"] != record["input_sha256"] or ecg_item["input"]["sha256"] != record["input_sha256"]:
            raise RuntimeError("SHA-256 sidecar не совпал с QC-манифестом")
        source = (data_root / record["relative_path"]).resolve()
        source.relative_to(data_root)
        if sha256_file(source) != record["input_sha256"]:
            raise RuntimeError(f"CSV изменился после QC: {record_id}")
        if record["montage_id"] != operator["montage_id"]:
            raise RuntimeError("Монтаж записи не совпал с принятой калибровкой ТТРКГ")
        frame = pd.read_csv(source, encoding="utf-8")
        time_s = pd.to_numeric(frame["TIME_s"], errors="raise").to_numpy(float)
        pulse_raw = pd.to_numeric(frame[operator["signal_column"]], errors="raise").to_numpy(float)
        base_raw = pd.to_numeric(frame[operator["base_column"]], errors="raise").to_numpy(float)
        pulse_ohm = float(operator["sign"]) * float(operator["gain"]) * float(operator["unit_scale_to_ohm"]) * pulse_raw
        base_ohm = float(operator["base_sign"]) * float(operator["base_gain"]) * base_raw + float(operator["base_offset_ohm"])
        for mode in ("задержка_вдох", "задержка_выдох"):
            left, right = map(float, breath["accepted_modes"][mode])
            margin = float(operator["hold_margin_s"])
            item = ensemble_waveform(time_s, pulse_ohm, ecg_item["rpeaks_s"], (left + margin, right - margin), raw_grid_s, operator["baseline_s"])
            mode_mask = (time_s >= left + margin) & (time_s <= right - margin)
            baseline_z = float(np.median(base_ohm[mode_mask]))
            if not np.isfinite(baseline_z) or baseline_z <= 0:
                raise RuntimeError("Базовый импеданс должен быть конечным и положительным")
            outputs.append({
                "experiment_id": "exp02", "subject_id": record["subject_id"],
                "record_id": record_id, "configuration_id": record["configuration_id"],
                "montage_id": record["montage_id"], "side_montage_id": record["side_montage_id"],
                "side_size_mm": int(record["side_size_mm"]),
                "channel_state": "ttrkg_channel_1_with_side_channel_2_connected",
                "mode": mode, "time_from_r_corrected_s": corrected_grid_s.tolist(),
                "mean_delta_z_ohm": item["mean"].tolist(),
                "se_within_record_ohm": item["se_within_record"].tolist(),
                "baseline_z_ohm": baseline_z,
                "mean_fractional_delta_z": (item["mean"] / baseline_z).tolist(),
                "n_beats": item["n_beats"], "input_sha256": record["input_sha256"],
                "breathing_sidecar_sha256": sha256_file(breath_path),
                "ecg_sidecar_sha256": sha256_file(ecg_path),
            })
    if len(outputs) != 2 * len(records):
        raise RuntimeError("Не получено ровно два дыхательных ансамбля на запись")
    artifact = {
        "schema_version": 2, "analysis": "40.01_exp02_ttrkg_ensembles",
        "status": "accepted_input_conditional_ttrkg_ensembles",
        "config_sha256": sha256_file(config_path), "record_manifest_sha256": sha256_file(manifest_path),
        "timing": {"status": "accepted", "group_delay_s": float(operator["group_delay_s"]), "grid": "physiological_time_relative_to_R"},
        "uncertainty_status": "within_record_se_only_not_total",
        "ensembles": outputs,
    }
    out_dir = derived_root / "exp02" / "analysis"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "40.01_ttrkg_ensembles.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("40.01 real_data_status: artifact_written", out_path)


## Граница результата

Выход — проверенные условные ансамбли конкретных записей, а не функция сердца.
Различия файлов нельзя автоматически приписывать приборному дрейфу: одновременно
менялись время, физиологическое состояние, контакт и боковая сборка.
